# BINN Gene Feature Extraction: Spatial Transcriptomics → Proteomics

This notebook uses a **Biology-Informed Neural Network (BINN)** to reduce ~18,000 genes from Visium HD spatial transcriptomics data to a compact, biologically meaningful subset. The selected genes are identified by their ability to predict spatial protein abundance (CODEX, 44 markers), and the final output is a filtered `.h5ad` file containing all original bins but only the selected genes.

## Pipeline

```
train_rna.h5ad  (N bins × 18,085 genes)       train_pro.h5ad  (N bins × 44 proteins)
       │                                                │
       ▼                                                │
  Preprocessing                                        │
  · In-tissue filter                                   │
  · QC (min counts)                                    │
  · HVG selection (3,000 genes)                        │
  · Subset → normalize → log1p                         │
  · Protein CLR normalization ◄────────────────────────┘
       │
       ▼
  Reactome Pathway Mapping
  · HGNC symbol → UniProt (MyGene.info API)
  · UniProt → Reactome pathway hierarchy
       │
       ▼
  BINN Training
  · Sparse masked layers following pathway hierarchy
  · Multi-task: predicts all 44 proteins simultaneously
  · Early stopping on validation MSE
       │
       ▼
  SHAP Feature Importance
  · GradientExplainer scores each gene's contribution
  · Aggregate across all protein outputs and samples
       │
       ▼
  Output: filtered_rna.h5ad
  · All original bins preserved
  · Gene axis reduced to selected subset
  · Raw counts + normalised expression retained
  · SHAP importance scores in .var
```

## Key Design Decisions

- **Pathway-constrained sparsity**: connections are biologically constrained by Reactome membership, preventing overfitting and ensuring interpretability.
- **Multi-task supervision**: training against all 44 proteins simultaneously ensures the selected genes are proteomically relevant, not just transcriptomically variable.
- **Memory-efficient batching**: all operations on the 166k-bin dataset are chunked to remain within 16 GB RAM.
- **Apply-to-any-split**: the gene selection is learned on the training set and applied identically to any other `.h5ad` sharing the same 18,085-gene space.


## 0. Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg, imp in [('anndata', 'anndata'), ('scikit-misc', 'skmisc'),
                 ('scanpy', 'scanpy'), ('binn', 'binn'), ('shap', 'shap')]:
    try:
        __import__(imp)
    except ImportError:
        print(f'Installing {pkg}...')
        _install(pkg)

print('All packages available.')


Installing anndata...
Installing scikit-misc...
Installing scanpy...
Installing binn...
All packages available.


In [2]:
import gc
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import requests
from tqdm.auto import tqdm

import anndata as ad
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import shap

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')


Device : cpu
PyTorch: 2.11.0+cpu


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuration

In [4]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR       = Path('/content/drive/MyDrive/IRBM_Data')
TRAIN_RNA_PATH = BASE_DIR / 'train_rna.h5ad'
TRAIN_PRO_PATH = BASE_DIR / 'train_pro.h5ad'
OUTPUT_DIR     = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BINN_DATA_DIR  = BASE_DIR / 'binn_data'

# ── Preprocessing ──────────────────────────────────────────────────────────────
MIN_COUNTS_GENE  = 5
NORMALIZE_TARGET = 1e4

# ── BINN architecture ──────────────────────────────────────────────────────────
N_BINN_LAYERS  = 4
DROPOUT_RATE   = 0.2

# ── Training ───────────────────────────────────────────────────────────────────
BATCH_SIZE      = 256
N_EPOCHS        = 100
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4
VAL_SPLIT       = 0.15
EARLY_STOP_PAT  = 15

# ── Gene selection — set one, leave other None ─────────────────────────────────
N_GENES_FIXED   = None   # e.g. 3000 to force a fixed output
CUMULATIVE_FRAC = 0.90   # data-driven: keep genes covering 90% of SHAP importance

print('Configuration loaded.')

Configuration loaded.


### 1.1 Reactome Reference Files

Run this cell once; files are saved to Google Drive and persist across sessions.

In [5]:
BINN_DATA_DIR.mkdir(parents=True, exist_ok=True)

_reactome_files = {
    'uniprot_2_reactome_2025_01_14.txt':
        'https://reactome.org/download/current/UniProt2Reactome.txt',
    'reactome_pathways_relation_2025_01_14.txt':
        'https://reactome.org/download/current/ReactomePathwaysRelation.txt',
    'reactome_pathways_names_2025_01_21.txt':
        'https://reactome.org/download/current/ReactomePathways.txt',
}

for fname, url in _reactome_files.items():
    dest = BINN_DATA_DIR / fname
    if not dest.exists():
        print(f'Downloading {fname} ...')
        os.system(f'wget -q "{url}" -O "{dest}"')
    else:
        print(f'Found (cached): {fname}')

found = list(BINN_DATA_DIR.glob('*.txt'))
print(f'\n{len(found)} Reactome files ready.')


Found (cached): uniprot_2_reactome_2025_01_14.txt
Found (cached): reactome_pathways_relation_2025_01_14.txt
Found (cached): reactome_pathways_names_2025_01_21.txt

3 Reactome files ready.


## 2. Data Loading and Preprocessing

All three functions are memory-efficient:
- `load_spatial_data` uses `backed='r'` (memory-mapped) to filter on obs metadata before pulling any matrix into RAM.
- `preprocess_rna` selects HVGs on raw sparse counts, subsets the gene axis **before** normalization, then normalizes only the small (N × 3,000) matrix.
- `preprocess_protein` performs CLR normalization in 500-row chunks so the dense slice is never larger than 500 × 44.

In [6]:
def load_spatial_data(rna_path, pro_path=None, in_tissue_only=True):
    """
    Load RNA (and optionally protein) AnnData objects.

    Uses memory-mapped access to filter on obs metadata before loading
    any expression matrix into RAM. Returns sparse CSR matrices.
    """
    print(f'Loading RNA from {rna_path} ...')
    rna = ad.read_h5ad(rna_path, backed='r')
    print(f'  Shape: {rna.shape}  (bins × genes)')

    rna_obs = (rna.obs_names[rna.obs['in_tissue'] == 1]
               if in_tissue_only and 'in_tissue' in rna.obs.columns
               else rna.obs_names)

    pro = None
    if pro_path is not None:
        print(f'Loading protein from {pro_path} ...')
        pro = ad.read_h5ad(pro_path, backed='r')
        print(f'  Shape: {pro.shape}  (bins × markers)')

        pro_obs = (pro.obs_names[pro.obs['in_tissue'] == 1]
                   if in_tissue_only and 'in_tissue' in pro.obs.columns
                   else pro.obs_names)

        common = rna_obs.intersection(pro_obs)
        print(f'  Common in-tissue bins: {len(common)}')

        rna = rna[common].to_memory()
        pro = pro[common].to_memory()
    else:
        rna = rna[rna_obs].to_memory()

    if not sp.issparse(rna.X):
        rna.X = sp.csr_matrix(rna.X)
    if pro is not None and not sp.issparse(pro.X):
        pro.X = sp.csr_matrix(pro.X)

    print(f'  RNA loaded: {rna.shape}')
    return rna, pro


def preprocess_rna(adata, normalize_target=1e4, min_counts_gene=5):
    """
    RNA preprocessing. filter_cells is omitted as in_tissue filtering
    in load_spatial_data already removes empty/off-tissue bins.
    filter_genes removes genes with negligible expression across all bins.
    """
    print('Preprocessing RNA ...')

    if not sp.issparse(adata.X):
        adata.X = sp.csr_matrix(adata.X)

    sc.pp.filter_genes(adata, min_counts=min_counts_gene)
    print(f'  After gene QC filter: {adata.shape}')

    sc.pp.normalize_total(adata, target_sum=normalize_target, inplace=True)
    sc.pp.log1p(adata, chunked=True, chunk_size=5000)
    adata.X = adata.X.tocsr()

    all_genes = adata.var_names.tolist()
    print(f'  Preprocessing complete: {adata.shape}')
    return adata, all_genes


def preprocess_protein(adata, chunk_size=500):
    """
    Centred log-ratio (CLR) normalisation for CODEX protein data.

    Processes the matrix in `chunk_size`-row slices so that only a small
    dense block is ever resident in RAM (chunk_size × n_markers).
    Z-scores are applied in-place after CLR.
    """
    print('Preprocessing protein (CLR normalisation) ...')

    n_bins, n_markers = adata.shape
    X_clr = np.empty((n_bins, n_markers), dtype=np.float32)

    for start in range(0, n_bins, chunk_size):
        end   = min(start + chunk_size, n_bins)
        chunk = adata.X[start:end]
        if sp.issparse(chunk):
            chunk = chunk.toarray()
        chunk  = chunk.astype(np.float32) + 1.0
        geom   = np.exp(np.mean(np.log(chunk), axis=1, keepdims=True))
        X_clr[start:end] = np.log(chunk / geom)

    mean_ = X_clr.mean(axis=0, keepdims=True)
    std_  = X_clr.std(axis=0,  keepdims=True) + 1e-8
    X_clr -= mean_
    X_clr /= std_

    adata      = adata.copy()   # copy obs/var metadata only
    adata.X    = X_clr
    adata.uns['protein_clr_mean'] = mean_
    adata.uns['protein_clr_std']  = std_

    print(f'  Protein matrix shape: {adata.X.shape}')
    return adata


In [8]:
rna_train, pro_train = load_spatial_data(TRAIN_RNA_PATH, TRAIN_PRO_PATH)

rna_train, all_genes = preprocess_rna(
    rna_train,
    normalize_target = NORMALIZE_TARGET,
    min_counts_gene  = MIN_COUNTS_GENE,
)

pro_train = preprocess_protein(pro_train)

common_obs = rna_train.obs_names.intersection(pro_train.obs_names)
if len(common_obs) < len(rna_train):
    print(f'Re-aligning after QC: {len(common_obs)} common bins')
    rna_train = rna_train[common_obs]
    pro_train = pro_train[common_obs]

print(f'\nFinal shapes:')
print(f'  RNA train    : {rna_train.shape}')
print(f'  Protein train: {pro_train.shape}')


Loading RNA from /content/drive/MyDrive/IRBM_Data/train_rna.h5ad ...
  Shape: (166186, 18085)  (bins × genes)
Loading protein from /content/drive/MyDrive/IRBM_Data/train_pro.h5ad ...
  Shape: (166186, 44)  (bins × markers)
  Common in-tissue bins: 166186
  RNA loaded: (166186, 18085)
Preprocessing RNA ...
  After gene QC filter: (166186, 18033)
  Preprocessing complete: (166186, 18033)
Preprocessing protein (CLR normalisation) ...
  Protein matrix shape: (166186, 44)

Final shapes:
  RNA train    : (166186, 18033)
  Protein train: (166186, 44)


## 3. Gene → Reactome Pathway Mapping

Converts HGNC gene symbols to Reactome pathway identifiers via two steps:
1. **MyGene.info REST API**: gene symbol → UniProt Swiss-Prot accession (batched, cached).
2. **Reactome flat files**: UniProt → pathway ID → parent pathway ID (bundled with the `binn` package).

The result is cached to disk so subsequent runs skip the API calls.

In [9]:
def fetch_gene_to_uniprot_batch(gene_symbols, batch_size=200):
    """
    Convert HGNC gene symbols to UniProt Swiss-Prot accessions via MyGene.info.

    Parameters
    ----------
    gene_symbols : list[str]
    batch_size   : int  — number of symbols per API request

    Returns
    -------
    dict[str, list[str]]  — {gene_symbol: [uniprot_id, ...]}
    """
    print(f'Querying MyGene.info for {len(gene_symbols)} gene symbols...')
    symbol_to_uniprot = {}
    url     = 'https://mygene.info/v3/query'
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}

    for i in tqdm(range(0, len(gene_symbols), batch_size), desc='Gene→UniProt'):
        batch   = gene_symbols[i : i + batch_size]
        payload = {'q': ','.join(batch), 'scopes': 'symbol',
                   'fields': 'uniprot', 'species': 'human', 'size': batch_size}
        try:
            resp = requests.post(url, data=payload, headers=headers, timeout=30)
            resp.raise_for_status()
            for hit in resp.json():
                sym = hit.get('query', '')
                if 'uniprot' in hit and 'Swiss-Prot' in hit['uniprot']:
                    acc = hit['uniprot']['Swiss-Prot']
                    symbol_to_uniprot[sym] = [acc] if isinstance(acc, str) else acc
        except Exception as e:
            print(f'  Warning: batch {i // batch_size} failed: {e}')

    print(f'  Mapped {len(symbol_to_uniprot)}/{len(gene_symbols)} genes to UniProt.')
    return symbol_to_uniprot


def build_gene_reactome_mapping(gene_symbols, binn_data_dir, cache_path=None):
    """
    Build the gene → Reactome pathway table used to define BINN connectivity.

    Results are written to `cache_path` (Parquet) on first run and loaded
    from cache on subsequent runs.

    Returns
    -------
    gene_to_pathway  : pd.DataFrame  — columns [input, translation, ...]
    pathway_relations: list[tuple]   — (child_pathway, parent_pathway)
    covered_genes    : list[str]     — gene symbols with at least one pathway
    """
    cache_path = cache_path or OUTPUT_DIR / 'gene_reactome_mapping.parquet'

    if Path(cache_path).exists():
        print(f'Loading cached mapping from {cache_path}')
        gene_to_pathway = pd.read_parquet(cache_path)
    else:
        up2r = pd.read_csv(
            binn_data_dir / 'uniprot_2_reactome_2025_01_14.txt',
            sep='\t', header=None,
            names=['input', 'translation', 'url', 'name', 'evidence', 'species'],
        )
        up2r_human = up2r[up2r['species'] == 'Homo sapiens']
        sym2up     = fetch_gene_to_uniprot_batch(gene_symbols)

        rows = []
        for sym, uids in sym2up.items():
            for uid in uids:
                matches = up2r_human[up2r_human['input'] == uid].copy()
                if len(matches):
                    matches['input'] = sym
                    rows.append(matches)

        if not rows:
            raise RuntimeError(
                'No genes mapped to Reactome. Check network connectivity.')

        gene_to_pathway = pd.concat(rows, ignore_index=True).drop_duplicates()
        gene_to_pathway.to_parquet(cache_path)
        print(f'Mapping saved to {cache_path}')

    p_rel = pd.read_csv(
        binn_data_dir / 'reactome_pathways_relation_2025_01_14.txt',
        sep='\t', header=None, names=['target', 'source'],
    )
    p_rel_human = p_rel[
        p_rel['target'].str.startswith('R-HSA') &
        p_rel['source'].str.startswith('R-HSA')
    ]
    pathway_relations = list(p_rel_human.itertuples(index=False, name=None))
    covered_genes     = gene_to_pathway['input'].unique().tolist()

    print(f'  Genes with Reactome mapping : {len(covered_genes)}')
    print(f'  Human pathway relations     : {len(pathway_relations)}')
    return gene_to_pathway, pathway_relations, covered_genes


In [10]:
gene_to_pathway_df, pathway_relations, covered_genes = build_gene_reactome_mapping(
    all_genes,           # <── was hvg_genes
    binn_data_dir = BINN_DATA_DIR,
)


Loading cached mapping from /content/drive/MyDrive/IRBM_Data/outputs/gene_reactome_mapping.parquet
  Genes with Reactome mapping : 1860
  Human pathway relations     : 2899


## 4. BINN Architecture

The BINN is a sparse feedforward network whose connectivity is constrained by the Reactome hierarchy:

```
Input genes  →  Level-1 pathways  →  Level-2 pathways  →  …  →  Top-level pathways  →  Protein predictions
```

Each layer is a `MaskedLinear` module — a standard linear layer whose weight matrix is elementwise-multiplied by a fixed binary mask encoding which gene–pathway (or pathway–pathway) connections are biologically supported. Connections not present in Reactome are permanently zeroed.

In [11]:
def build_pathway_layers(gene_symbols_subset, gene_to_pathway_df,
                         pathway_relations, n_layers):
    """
    Construct sparse connectivity masks for each BINN layer.

    Parameters
    ----------
    gene_symbols_subset : list[str]  — BINN input genes (HVG ∩ Reactome)
    gene_to_pathway_df  : pd.DataFrame
    pathway_relations   : list[tuple] — (child, parent) pathway pairs
    n_layers            : int         — number of hierarchy levels to traverse

    Returns
    -------
    layer_nodes : list[list[str]]        — node names per layer
    masks       : list[torch.BoolTensor] — connectivity masks
    """
    import networkx as nx

    G   = nx.DiGraph()
    for child, parent in pathway_relations:
        G.add_edge(child, parent)

    g2p = gene_to_pathway_df[
        gene_to_pathway_df['input'].isin(gene_symbols_subset)
    ]
    gene_to_l1 = g2p.groupby('input')['translation'].apply(list).to_dict()

    layer_nodes = [gene_symbols_subset]
    layer_sets  = [set(gene_symbols_subset)]

    current = set()
    for g in gene_symbols_subset:
        current.update(gene_to_l1.get(g, []))

    for depth in range(n_layers):
        layer_nodes.append(sorted(current))
        layer_sets.append(current)

        nxt = set()
        for p in current:
            nxt.update(G.successors(p))
        nxt -= layer_sets[1]
        current = nxt
        if not current:
            print(f'  Hierarchy exhausted at layer {depth + 1}')
            break

    print(f'Layer sizes: {[len(l) for l in layer_nodes]}')

    masks    = []
    gene_idx = {g: i for i, g in enumerate(layer_nodes[0])}
    l1_idx   = {p: i for i, p in enumerate(layer_nodes[1])}

    mask0 = torch.zeros(len(layer_nodes[1]), len(layer_nodes[0]), dtype=torch.bool)
    for g, pathways in gene_to_l1.items():
        if g in gene_idx:
            for p in pathways:
                if p in l1_idx:
                    mask0[l1_idx[p], gene_idx[g]] = True
    masks.append(mask0)

    for d in range(1, len(layer_nodes) - 1):
        src = {n: i for i, n in enumerate(layer_nodes[d])}
        dst = {n: i for i, n in enumerate(layer_nodes[d + 1])}
        m   = torch.zeros(len(layer_nodes[d + 1]), len(layer_nodes[d]), dtype=torch.bool)
        for child, parent in pathway_relations:
            if child in src and parent in dst:
                m[dst[parent], src[child]] = True
        masks.append(m)

    for i, m in enumerate(masks):
        density = m.float().mean().item() * 100
        print(f'  Mask {i}: shape={tuple(m.shape)}  sparsity={100 - density:.1f}%')

    return layer_nodes, masks


class MaskedLinear(nn.Module):
    """Linear layer with a fixed binary pathway-connectivity mask."""

    def __init__(self, in_features, out_features, mask: torch.Tensor, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.register_buffer('mask', mask.float())
        nn.init.kaiming_uniform_(self.weight, a=0.01)

    def forward(self, x):
        return F.linear(x, self.weight * self.mask, self.bias)


class BINNEncoder(nn.Module):
    """
    Biology-Informed Neural Network encoder.

    Encodes gene expression through the Reactome pathway hierarchy.
    The `.encode()` method returns the bottleneck (top-level pathway)
    representation; `.forward()` additionally passes it through a small
    MLP head to predict protein abundances.
    """

    def __init__(self, masks, n_outputs, dropout=0.2):
        super().__init__()
        self.pathway_layers = nn.ModuleList()
        self.batch_norms    = nn.ModuleList()
        self.dropouts       = nn.ModuleList()

        for mask in masks:
            out_dim, in_dim = mask.shape
            self.pathway_layers.append(MaskedLinear(in_dim, out_dim, mask))
            self.batch_norms.append(nn.BatchNorm1d(out_dim))
            self.dropouts.append(nn.Dropout(p=dropout))

        bottleneck_dim = masks[-1].shape[0]
        self.head = nn.Sequential(
            nn.Linear(bottleneck_dim, 256), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_outputs),
        )

    def encode(self, x):
        for linear, bn, drop in zip(self.pathway_layers, self.batch_norms, self.dropouts):
            x = drop(torch.tanh(bn(linear(x))))
        return x

    def forward(self, x):
        return self.head(self.encode(x))


In [12]:
binn_input_genes = [g for g in all_genes if g in covered_genes]
print(f'Genes entering BINN: {len(binn_input_genes)} / {len(all_genes)} total QC-passing genes')

layer_nodes, masks = build_pathway_layers(
    binn_input_genes, gene_to_pathway_df, pathway_relations, N_BINN_LAYERS
)

masks_device = [m.to(DEVICE) for m in masks]
n_proteins   = pro_train.shape[1]
model        = BINNEncoder(masks_device, n_outputs=n_proteins, dropout=DROPOUT_RATE).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')
print(model)


Genes entering BINN: 1860 / 18033 total QC-passing genes
Layer sizes: [1860, 1531, 140, 47, 15]
  Mask 0: shape=(1531, 1860)  sparsity=99.8%
  Mask 1: shape=(140, 1531)  sparsity=99.9%
  Mask 2: shape=(47, 140)  sparsity=99.3%
  Mask 3: shape=(15, 47)  sparsity=97.7%

Trainable parameters: 3,089,888
BINNEncoder(
  (pathway_layers): ModuleList(
    (0-3): 4 x MaskedLinear()
  )
  (batch_norms): ModuleList(
    (0): BatchNorm1d(1531, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(140, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): BatchNorm1d(47, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): BatchNorm1d(15, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropouts): ModuleList(
    (0-3): 4 x Dropout(p=0.2, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=15, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear

## 5. DataLoaders

The full expression matrices are loaded into CPU RAM as float32 tensors once, then served to the GPU in `BATCH_SIZE`-row batches by PyTorch `DataLoader`. A 15% random split is held out as an internal validation set for early stopping.

In [13]:
def _to_tensor(adata, gene_list=None):
    """Extract a float32 tensor from an AnnData, optionally subsetting genes."""
    obj = adata[:, [g for g in gene_list if g in adata.var_names]] \
          if gene_list is not None else adata
    X = obj.X
    if sp.issparse(X):
        X = X.toarray()
    return torch.tensor(X.astype(np.float32))


def make_dataloaders(rna_adata, pro_adata, binn_genes,
                     batch_size=256, val_split=0.15, seed=42):
    """
    Build train / validation DataLoaders from aligned AnnData objects.

    Genes in `binn_genes` that are absent from the RNA data are padded
    with zeros to preserve mask alignment.
    """
    X_rna = _to_tensor(rna_adata, gene_list=binn_genes)
    X_pro = _to_tensor(pro_adata)

    present = [g for g in binn_genes if g in rna_adata.var_names]
    if len(present) < len(binn_genes):
        print(f'  Padding {len(binn_genes) - len(present)} missing BINN genes with zeros.')
        full  = torch.zeros(X_rna.shape[0], len(binn_genes))
        g2col = {g: i for i, g in enumerate(binn_genes)}
        for i, g in enumerate(present):
            full[:, g2col[g]] = X_rna[:, i]
        X_rna = full

    print(f'  RNA tensor : {tuple(X_rna.shape)}')
    print(f'  Pro tensor : {tuple(X_pro.shape)}')

    dataset = TensorDataset(X_rna, X_pro)
    n_val   = int(len(dataset) * val_split)
    n_train = len(dataset) - n_val
    gen     = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=gen)

    pin = DEVICE.type == 'cuda'
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2,
                              shuffle=False, num_workers=0, pin_memory=pin)

    print(f'  Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
    return train_loader, val_loader


print('Building DataLoaders...')
train_loader, val_loader = make_dataloaders(
    rna_train, pro_train,
    binn_genes  = binn_input_genes,
    batch_size  = BATCH_SIZE,
    val_split   = VAL_SPLIT,
)


Building DataLoaders...
  RNA tensor : (166186, 1860)
  Pro tensor : (166186, 44)
  Train batches: 552 | Val batches: 49


## 6. Training

In [14]:
def train_binn(model, train_loader, val_loader, n_epochs, lr,
               weight_decay, early_stop_patience, device):
    """
    Train the BINN with MSE loss against all protein targets simultaneously.

    Uses AdamW optimiser with cosine annealing LR schedule and
    early stopping based on validation MSE.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'val_pearson'
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=lr * 0.01)
    criterion = nn.MSELoss()

    history      = {'train_loss': [], 'val_loss': [], 'val_pearson': []}
    best_val     = float('inf')
    best_state   = None
    patience_ctr = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_losses = []
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), Yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses, preds, targets = [], [], []
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                Yp = model(Xb)
                val_losses.append(criterion(Yp, Yb).item())
                preds.append(Yp.cpu().numpy())
                targets.append(Yb.cpu().numpy())

        preds   = np.vstack(preds)
        targets = np.vstack(targets)
        pearson = np.nanmean([
            np.corrcoef(preds[:, j], targets[:, j])[0, 1]
            for j in range(preds.shape[1])
        ])

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(np.mean(val_losses))
        history['val_pearson'].append(pearson)
        scheduler.step()

        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:>4d}/{n_epochs}  '
                  f'train={history["train_loss"][-1]:.4f}  '
                  f'val={history["val_loss"][-1]:.4f}  '
                  f'pearson={pearson:.3f}')

        if history['val_loss'][-1] < best_val:
            best_val   = history['val_loss'][-1]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= early_stop_patience:
                print(f'Early stopping at epoch {epoch}.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f'Best validation MSE: {best_val:.4f}')
    return history


In [ ]:
print('Training BINN...')
history = train_binn(
    model, train_loader, val_loader,
    n_epochs            = N_EPOCHS,
    lr                  = LEARNING_RATE,
    weight_decay        = WEIGHT_DECAY,
    early_stop_patience = EARLY_STOP_PAT,
    device              = DEVICE,
)

torch.save(model.state_dict(), OUTPUT_DIR / 'binn_checkpoint.pt')
print('Checkpoint saved.')


Training BINN...
Epoch    1/100  train=1.0002  val=0.9990  pearson=0.079
Epoch   10/100  train=0.9861  val=1.1414  pearson=0.123
Epoch   20/100  train=0.9833  val=0.9704  pearson=0.150


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE loss')
axes[0].set_title('Training loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['val_pearson'], color='steelblue')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Mean Pearson r')
axes[1].set_title('Validation Pearson r (mean across proteins)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. SHAP-Based Gene Importance

`GradientExplainer` computes per-gene importance scores by propagating gradients from all 44 protein outputs back to the gene input layer. The final importance of each gene is the mean absolute SHAP value across all protein outputs and all explained samples.

The returned array has shape `(n_samples, n_genes, n_outputs)` — this cell reduces it to a single 1-D importance vector of length `n_genes`.

In [ ]:
def compute_gene_shap_importance(model, train_loader,
                                 n_background=512, n_explain=1024,
                                 device=DEVICE):
    """
    Compute per-gene SHAP importance scores using GradientExplainer.

    Parameters
    ----------
    model        : trained BINNEncoder
    train_loader : DataLoader  — source of background and explain samples
    n_background : int  — number of reference samples for the explainer
    n_explain    : int  — number of samples to explain
    device       : torch.device

    Returns
    -------
    importance : np.ndarray, shape (n_genes,)
        Mean |SHAP| aggregated over all protein outputs and explained samples.
    """
    model.eval()

    all_X = []
    for Xb, _ in train_loader:
        all_X.append(Xb)
        if sum(x.shape[0] for x in all_X) >= n_background + n_explain:
            break
    all_X      = torch.cat(all_X, dim=0)
    background = all_X[:n_background].to(device)
    explain_X  = all_X[n_background : n_background + n_explain].to(device)

    print(f'GradientExplainer: {explain_X.shape[0]} samples, '
          f'{background.shape[0]} background ...')

    explainer   = shap.GradientExplainer(model, background)
    shap_values = explainer.shap_values(explain_X)
    # shap_values shape: (n_outputs, n_samples, n_genes)  or  (n_samples, n_genes, n_outputs)
    sv = np.array(shap_values)
    print(f'  Raw SHAP array shape: {sv.shape}')

    # Normalise to (n_genes,): mean |shap| over all outputs and samples
    # Handle both possible axis orderings
    if sv.ndim == 3:
        # (n_outputs, n_samples, n_genes) → mean over axes 0,1
        if sv.shape[0] == explain_X.shape[0]:
            # actually (n_samples, n_genes, n_outputs)
            importance = np.abs(sv).mean(axis=(0, 2))
        else:
            importance = np.abs(sv).mean(axis=(0, 1))
    else:
        importance = np.abs(sv).mean(axis=0)

    importance = importance.flatten()
    assert importance.shape[0] == explain_X.shape[1], (
        f'Shape mismatch: importance={importance.shape}, '
        f'n_genes={explain_X.shape[1]}')

    print(f'  Importance shape: {importance.shape}')
    return importance


gene_shap = compute_gene_shap_importance(
    model, train_loader,
    n_background = min(512,  len(train_loader.dataset) // 4),
    n_explain    = min(1024, len(train_loader.dataset) // 2),
    device       = DEVICE,
)


## 8. Gene Selection

In [ ]:
importance_df = pd.DataFrame({
    'gene':            binn_input_genes,
    'shap_importance': gene_shap,
}).sort_values('shap_importance', ascending=False).reset_index(drop=True)

importance_df['shap_rank'] = range(1, len(importance_df) + 1)

importance_df.to_csv(OUTPUT_DIR / 'gene_shap_importance.csv', index=False)
print('Top 20 genes by SHAP importance:')
print(importance_df.head(20).to_string(index=False))


In [ ]:
# Cumulative importance curve
cum = np.cumsum(importance_df['shap_importance'].values)
cum = cum / cum[-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

top40 = importance_df.head(40)
axes[0].barh(top40['gene'][::-1], top40['shap_importance'][::-1],
             color=cm.viridis(np.linspace(0.2, 0.9, 40)))
axes[0].set_xlabel('Mean |SHAP|')
axes[0].set_title('Top 40 genes by SHAP importance')
axes[0].tick_params(axis='y', labelsize=7)

axes[1].plot(range(1, len(cum) + 1), cum, lw=1.5)
for frac in [0.5, 0.8, 0.9]:
    n = np.searchsorted(cum, frac) + 1
    axes[1].axhline(frac, color='red', ls='--', alpha=0.4)
    axes[1].axvline(n,    color='red', ls='--', alpha=0.4)
    axes[1].text(n + 2, frac - 0.04, f'{n} genes\n({int(frac*100)}%)', fontsize=8)
axes[1].set_xlabel('Genes (ranked)')
axes[1].set_ylabel('Cumulative SHAP importance')
axes[1].set_title('Cumulative importance'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gene_importance.png', dpi=150, bbox_inches='tight')
plt.show()

for frac in [0.5, 0.8, 0.9, 0.95]:
    n = np.searchsorted(cum, frac) + 1
    print(f'  {int(frac*100)}% importance → top {n} genes')


In [ ]:
if N_GENES_FIXED is not None:
    SELECTED_GENES = importance_df.head(N_GENES_FIXED)['gene'].tolist()
    print(f'Gene selection: top {N_GENES_FIXED} (fixed count)')
else:
    total   = importance_df['shap_importance'].sum()
    cum_sum = importance_df['shap_importance'].cumsum()
    n_auto  = int((cum_sum <= total * CUMULATIVE_FRAC).sum()) + 1
    SELECTED_GENES = importance_df.head(n_auto)['gene'].tolist()
    print(f'Gene selection: top {n_auto} genes '
          f'({CUMULATIVE_FRAC*100:.0f}% cumulative importance)')

print(f'Selected {len(SELECTED_GENES)} genes.')
print('Preview:', SELECTED_GENES[:10])


## 9. Output: Filtered h5ad

The output file preserves the **complete original AnnData structure** — all bins, all obs metadata (`array_row`, `array_col`, `pxl_*`, `in_tissue`, spatial coordinates) — but reduces the gene axis to the selected subset only.

| Layer / slot | Contents |
|---|---|
| `.X` | log-normalised expression (bins × selected genes) |
| `.layers['counts']` | raw UMI counts (bins × selected genes) |
| `.obs` | all original bin metadata unchanged |
| `.var` | gene names + `shap_importance` + `shap_rank` |
| `.uns['selected_genes']` | ordered list of selected gene names |
| `.uns['n_input_genes']` | original gene count (18,085) |

The function `write_filtered_h5ad` is designed to be called on **any** `.h5ad` that shares the same gene space (same 18,085 genes), so the identical gene selection can be applied to validation or test splits without re-running the BINN.

In [ ]:
def write_filtered_h5ad(source_path, selected_genes, output_path,
                        normalize_target=1e4, importance_df=None):
    """
    Load an h5ad file and write a filtered copy containing only `selected_genes`.

    All bins (obs) and all obs metadata are preserved unchanged.
    Raw counts are stored in `.layers['counts']`; `.X` holds log-normalised values.
    SHAP importance scores are stored in `.var` if `importance_df` is provided.

    Parameters
    ----------
    source_path      : Path  — original .h5ad (must share the same gene space)
    selected_genes   : list[str]
    output_path      : Path
    normalize_target : float
    importance_df    : pd.DataFrame or None  — must have 'gene' and 'shap_importance'
    """
    print(f'Reading {source_path} ...')
    src = ad.read_h5ad(source_path, backed='r')
    print(f'  Original shape: {src.shape}')

    present = [g for g in selected_genes if g in src.var_names]
    missing = len(selected_genes) - len(present)
    if missing:
        print(f'  Warning: {missing} selected genes absent from this file — skipped.')

    # Slice gene axis; bring into RAM
    out = src[: , present].to_memory()

    # Store raw counts before normalisation
    out.layers['counts'] = (out.X.copy() if sp.issparse(out.X)
                            else sp.csr_matrix(out.X))

    # Normalise
    sc.pp.normalize_total(out, target_sum=normalize_target, inplace=True)
    sc.pp.log1p(out)

    # Annotate .var with SHAP scores if available
    if importance_df is not None:
        imp = importance_df.set_index('gene')
        out.var['shap_importance'] = [
            imp.loc[g, 'shap_importance'] if g in imp.index else np.nan
            for g in out.var_names
        ]
        out.var['shap_rank'] = [
            int(imp.loc[g, 'shap_rank']) if g in imp.index else -1
            for g in out.var_names
        ]

    # Provenance metadata
    out.uns['selected_genes']  = present
    out.uns['n_input_genes']   = src.shape[1]
    out.uns['n_selected_genes'] = len(present)

    out.write_h5ad(output_path)
    size_mb = output_path.stat().st_size / 1e6
    print(f'  Written: {output_path}  ({size_mb:.1f} MB)')
    print(f'  Genes: {src.shape[1]:,} → {len(present):,}  '
          f'({100 * len(present) / src.shape[1]:.1f}% retained)')
    return out


In [ ]:
filtered_train = write_filtered_h5ad(
    source_path      = TRAIN_RNA_PATH,
    selected_genes   = SELECTED_GENES,
    output_path      = OUTPUT_DIR / 'train_rna_filtered.h5ad',
    normalize_target = NORMALIZE_TARGET,
    importance_df    = importance_df,
)

print('\nOutput AnnData structure:')
print(filtered_train)


## 10. Save Model and Summary

In [ ]:
# Save BINN model checkpoint with full metadata
torch.save({
    'model_state'     : model.state_dict(),
    'binn_input_genes': binn_input_genes,
    'selected_genes'  : SELECTED_GENES,
    'layer_node_names': [list(l) for l in layer_nodes],
    'n_proteins'      : n_proteins,
    'dropout'         : DROPOUT_RATE,
    'training_config' : {
        'N_TOP_GENES'  : N_TOP_GENES,
        'N_BINN_LAYERS': N_BINN_LAYERS,
        'N_EPOCHS'     : N_EPOCHS,
        'BATCH_SIZE'   : BATCH_SIZE,
        'LEARNING_RATE': LEARNING_RATE,
    },
}, OUTPUT_DIR / 'binn_model.pt')

# Save selected gene list
importance_df[importance_df['gene'].isin(SELECTED_GENES)].to_csv(
    OUTPUT_DIR / 'selected_genes.csv', index=False
)

print('=' * 55)
print('BINN FEATURE EXTRACTION — SUMMARY')
print('=' * 55)
print(f'Input genes (all)          : {rna_train.shape[1] + (N_TOP_GENES - len(hvg_genes)):,}')
print(f'Highly variable genes      : {len(hvg_genes):,}')
print(f'Genes with Reactome mapping: {len(covered_genes):,}')
print(f'BINN input genes           : {len(binn_input_genes):,}')
print(f'Selected output genes      : {len(SELECTED_GENES):,}')
print(f'Best val MSE               : {min(history["val_loss"]):.4f}')
print(f'Best val Pearson r         : {max(history["val_pearson"]):.3f}')
print()
print('Output files:')
for fname in ['train_rna_filtered.h5ad', 'selected_genes.csv',
              'gene_shap_importance.csv', 'binn_model.pt',
              'training_curves.png', 'gene_importance.png']:
    p = OUTPUT_DIR / fname
    if p.exists():
        print(f'  {fname}  ({p.stat().st_size / 1e6:.1f} MB)')


In [ ]:
print(f'Selected {len(SELECTED_GENES)} genes from {len(binn_input_genes)} BINN-input genes\n')

selected_df = importance_df[importance_df['gene'].isin(SELECTED_GENES)].copy()
print(selected_df[['shap_rank', 'gene', 'shap_importance', 'hvg_rank']].to_string(index=False))

## Appendix: Applying Gene Selection to Other Splits

Because `write_filtered_h5ad` only requires the source `.h5ad` and the `SELECTED_GENES` list, the same gene selection can be applied to any split that shares the same 18,085-gene space — no retraining required.

```python
# Example: apply to validation set
write_filtered_h5ad(
    source_path    = BASE_DIR / 'valid_rna.h5ad',
    selected_genes = SELECTED_GENES,
    output_path    = OUTPUT_DIR / 'valid_rna_filtered.h5ad',
    importance_df  = importance_df,
)

# Example: apply to test set
write_filtered_h5ad(
    source_path    = BASE_DIR / 'test_rna.h5ad',
    selected_genes = SELECTED_GENES,
    output_path    = OUTPUT_DIR / 'test_rna_filtered.h5ad',
    importance_df  = importance_df,
)
```

To reload the filtered file and inspect it:

```python
import anndata as ad
adata = ad.read_h5ad('outputs/train_rna_filtered.h5ad')

# Log-normalised expression matrix
X_norm = adata.X                          # (n_bins, n_selected_genes)

# Raw counts
X_raw  = adata.layers['counts']          # (n_bins, n_selected_genes)

# Spatial coordinates
coords = adata.obs[['array_row', 'array_col']]

# Selected gene names with importance
print(adata.var[['shap_importance', 'shap_rank']].head(10))
```
